# Lausanne: joint fleet simulation and sensor portfolios

One daily experiment combines **Bus lines 1, 9, 21, 33 and 54 as one fleet, Postal and Taxi** in the same task-driven event kernel. Follow the configuration cells, inspect means over joint replications, then compare sensor-count portfolios.

Run the cells in order with **Python 3.12** and the package's `optimization` and `notebook` extras (installation commands are in the repository README). The local Lausanne example bundle supplies all inputs. Unchanged settings reuse its verified results; editing settings automatically invokes the same simulation/analysis services as the web app. Fresh full-day computation is substantially slower than viewing precomputed results.

Figures appear inline. This notebook does not export datasets or images and its committed outputs are empty. The backend uses a temporary workspace, removed by the final cell or when the kernel exits.

In [ ]:
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "This tutorial requires Python 3.12. Select the "
        "'Mobile Sensing (Python 3.12)' kernel and restart the notebook."
    )

# Prefer this checkout when the notebook is opened from the repository. An
# installed wheel remains valid when no checkout is present.
repository = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "mobile_sensing").is_dir()
    ),
    None,
)
if repository is not None:
    source_root = str(repository / "src")
    if source_root not in sys.path:
        sys.path.insert(0, source_root)

from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mobile_sensing.application.project_models import (
    DemandEditor, DispatchEditor, FleetEditor, NumericRange, PortfolioEditor,
    PortfolioFleetEditor, ProjectConfig, ShiftGroup, SpatialFeatureWeight, SupplyEditor, TemporalInterval,
)
from mobile_sensing.application.run_models import RunOptions
from mobile_sensing.application.tutorial_workflow import (
    open_tutorial, simulation_result, analysis_result, fleet_results,
    portfolio_results, selected_frontier_points, portfolio_map,
)
from mobile_sensing.application.tutorial_plots import (
    plot_coverage, plot_daily_profiles, plot_frontiers, plot_frontier_maps, display_figure,
)

pd.options.display.float_format = '{:.5f}'.format
plt.rcParams.update({'font.size': 10, 'figure.dpi': 110, 'axes.titleweight': 'medium'})
temporary = TemporaryDirectory(prefix='mobile-sensing-lausanne-')
workspace = Path(temporary.name)
bundle, source_run, source_analysis = open_tutorial(workspace)
options = RunOptions(memory_limit_bytes=8 * 1024**3, job_timeout_s=14400)

## 1. Shared environment and observation window

The prepared environment covers all 28 Lausanne boundary features: **17,553 cells at 100 m**, including zero-population and zero-exposure cells. Spatial sampling uses 2024 population; routing uses the supplied directed road network with an assumed 30 km/h speed. Distance calculations use EPSG:2056.

The observation window is **14 January 2026, 00:00–24:00, Europe/Zurich**, with one-hour reporting bins and **50 joint replications**. The event kernel runs at exact event times; one hour is the reporting/aggregation resolution, not a one-hour movement jump. GTFS carry-in trips execute before midnight and are excluded from reported sensing.

In [ ]:
simulation = source_run.config.simulation.model_copy(update={
    'replications': 50,
    'start_time': '00:00', 'end_time': '24:00',
    'temporal_resolution_minutes': 60.0,
    'seed': 20260114,
})
bus = next(fleet for fleet in source_run.config.fleets if fleet.fleet_id == 'bus')

## 2. Demand, supply and dispatch

**Bus** combines complete GTFS trips on lines **1, 9, 21, 33 and 54** in one fleet with Scheduled dispatch and inferred duties. These are inferred vehicles on the combined service-day timeline, not a real registration inventory. The immutable example retains route identities and complete stop order.

**Postal** uses exactly 2,000 offline location tasks, released together at 06:00, with an 80:20 mixture of normalized population and public-service features. Twenty vehicles start uniformly between 06:00 and 12:00 and each works eight hours within 06:00–21:00. Four Auto service areas are generated from this expected spatial distribution before replications. Every area first receives one vehicle and the remaining fixed catalog is assigned by largest remainder of expected demand. The partition and catalog assignments remain frozen across all 50 replications; the Lausanne railway-station depot does not determine area membership.

Every task consumes one capacity unit; each vehicle holds 100, so one load serves at most 100 tasks. One-shot plans multiple trips within each shift, including 120 seconds per task, a minimum fifteen-minute depot stay for replenishment, and the final return. Vehicles start fully loaded. Remaining stock resets only after replenishment completes; the number of optional reloads is bounded. Tasks at the same snapped location are preferentially consecutive, but retain separate service times and consumption.

In [ ]:
postal = FleetEditor(
    fleet_id='postal', name='Postal',
    demand=DemandEditor(
        volume_mode='fixed', task_volume=2000.0, start_time='06:00', end_time='21:00',
        spatial_feature='population', spatial_weights=(
            SpatialFeatureWeight(feature='population', weight=0.8),
            SpatialFeatureWeight(feature='public_services', weight=0.2),
        ), location_condition='depot_roundtrip',
        release_mode='at_start', service_seconds=120.0, quantity=1.0,
    ),
    supply=SupplyEditor(
        fleet_size=20, operating_start='06:00', operating_end='21:00',
        activation='uniform_bounded', latest_start='12:00', work_hours=8.0,
        initial_location='depot', synthetic_depot=True,
        depot_longitude=6.6290923032, depot_latitude=46.5167918355,
        spatial_feature='population', post_service='return_after_plan',
        capacity_mode='consumable', capacity=100.0, depot_min_stay_minutes=15.0,
        service_area_mode='auto', auto_service_area_count=4,
    ),
    dispatch=DispatchEditor(
        mode='one_shot', max_cost_pairs=5_000_000, planning_timeout_seconds=600.0,
    ),
)

**Taxi** injects requests online. Its total is Poisson with expectation 800 per day, including explicit 21:00–24:00 demand. Origins and destinations independently use a 50:30:20 mixture of normalized population, commercial and public-service features, conditioned on distinct routable nodes and directed reachability.

Supply stays fixed at 80 physical vehicles: 10% activate at 00:00 and 30% each during 05:00–06:00, 09:00–11:00 and 15:00–16:00. Every shift lasts eight hours; the latest activation is 16:00. The experiment starts with no preceding-day Taxi shifts. Sequential nearest matching uses a 15-minute maximum pickup time; pickup takes 60 seconds and drop-off 30 seconds. Idle taxis cruise randomly on reachable directed roads.

In [ ]:
release_pattern = [
    ('00:00', '06:00', 0.05), ('06:00', '09:00', 0.18),
    ('09:00', '12:00', 0.13), ('12:00', '16:00', 0.20),
    ('16:00', '19:00', 0.24), ('19:00', '21:00', 0.12),
    ('21:00', '24:00', 0.08),
]
activation_groups = [
    ('Night', 8, '00:00', '00:00'),
    ('Morning', 24, '05:00', '06:00'),
    ('Daytime', 24, '09:00', '11:00'),
    ('Afternoon', 24, '15:00', '16:00'),
]
taxi = FleetEditor(
    fleet_id='taxi', name='Taxi',
    demand=DemandEditor(
        task_type='od', volume_mode='expected', task_volume=800.0,
        generation_timing='online', start_time='00:00', end_time='24:00',
        temporal_mode='shares',
        time_profile=tuple(TemporalInterval(start_time=a, end_time=b, value=s)
                           for a, b, s in release_pattern),
        spatial_feature='population', spatial_weights=(
            SpatialFeatureWeight(feature='population', weight=0.5),
            SpatialFeatureWeight(feature='commercial', weight=0.3),
            SpatialFeatureWeight(feature='public_services', weight=0.2),
        ), destination_feature='population', destination_spatial_weights=(
            SpatialFeatureWeight(feature='population', weight=0.5),
            SpatialFeatureWeight(feature='commercial', weight=0.3),
            SpatialFeatureWeight(feature='public_services', weight=0.2),
        ),
        pickup_seconds=60.0, service_seconds=30.0,
    ),
    supply=SupplyEditor(
        fleet_size=80, operating_start='00:00', operating_end='24:00',
        activation='uniform_bounded', latest_start='16:00', work_hours=8.0,
        spatial_feature='population', post_service='random_cruise',
        capacity_mode='occupancy', capacity=1.0,
        shift_groups=tuple(ShiftGroup(name=n, count=c, start_time=a, latest_start=b,
                                     work_hours=8.0) for n, c, a, b in activation_groups),
    ),
    dispatch=DispatchEditor(mode='sequential', max_pickup_minutes=15.0),
)
configuration = source_run.config.model_copy(update={
    'fleets': (bus, postal, taxi), 'simulation': simulation,
})
configuration = ProjectConfig.model_validate_json(configuration.model_dump_json())

## 3. Run the joint simulation

The following call automatically reuses matching verified results, or computes new operations and exposure when settings change. A portfolio never changes physical supply. All fleet means below use the same complete replication set; inactive vehicles remain in the catalog and contribute zeros.

In [ ]:
run = simulation_result(workspace, configuration, source_run, options=options)
fleet_views = {fleet.fleet_id: fleet_results(workspace, run, fleet.fleet_id)
               for fleet in configuration.fleets}
rows = [view[0]['fleets'][0] for view in fleet_views.values()]
display(pd.DataFrame(rows).set_index('fleet_id')[[
    'catalog_size', 'mean_released_tasks', 'mean_completed_tasks', 'mean_completion_rate',
]].rename(columns={
    'catalog_size': 'Vehicles', 'mean_released_tasks': 'Mean task releases',
    'mean_completed_tasks': 'Mean completed tasks', 'mean_completion_rate': 'Completion fraction',
}))

### Fleet totals and average physical vehicle

Left: sum vehicle exposure within a replication, then average replications. Right: divide that mean by the fleet's full catalog size. Each column shares one color scale across fleets; light gray denotes zero sensing. Maps show **sensing duration**, not a probability of coverage or service demand. All study cells remain in the denominator.

In [ ]:
figure = plot_coverage(fleet_views)
display_figure(figure)
plt.close(figure)

### Daily sensing, task releases and vehicle activity

All curves are replication means in local one-hour bins. Postal releases appear together at 06:00 because tasks are fixed and released at start. Active vehicles are time-weighted availability/operational presence; sensing includes operating movement, service, waiting and idling, but excludes stationary depot stays and off-duty time. Bus duty inference marks an entire unassigned idle interval of at least 60 minutes as off duty. Each panel uses its own vertical scale.

In [ ]:
figure = plot_daily_profiles(fleet_views)
display_figure(figure)
plt.close(figure)

## 4. Compare sensor-count portfolios

Keep the simulated physical fleets fixed. Enumerate counts in steps of **five sensors per fleet** without exceeding each catalog, with sensor budgets **0 to 50 in steps of five** and unit cost one sensor per equipped vehicle. Use **200 fleet sampling runs**, each drawing one joint replication and uniform subsets of physical vehicles. Simulation exposure is reported in one-hour bins; portfolio utility sums those bins into one **24-hour utility interval** and applies five-minute exponential saturation.

The frontier maximizes mean utility and **P05**, the linearly interpolated 5% quantile. P05 is a lower-tail outcome, not the average of the worst 5% nor a guaranteed minimum. With 200 draws its tail estimate remains conditional on the retained 50 replications; more sampling runs do not add operational replications.

In [ ]:
portfolio_editor = PortfolioEditor(
    source_run_id=run.run_id, sampling_runs=200, seed=20260115,
    cost_unit='sensor', budgets=tuple(float(b) for b in range(0, 51, 5)),
    risk_metric='p05', utility='exponential', saturation_minutes=5.0,
    utility_temporal_resolution_minutes=1440.0,
    spatial_weight='population',
    fleets=tuple(PortfolioFleetEditor(
        fleet_id=fleet, unit_cost=1.0,
        count_range=NumericRange(minimum=0, maximum=count - count % 5, step=5),
    ) for fleet, count in sorted(run.vehicle_counts.items())),
)

In [ ]:
SHOW_NONDOMINATED_ONLY = True
analysis = analysis_result(workspace, portfolio_editor, source_analysis, options=options)
frontiers = [portfolio_results(workspace, analysis, budget=budget)
             for budget in portfolio_editor.budgets]
figure = plot_frontiers(frontiers, nondominated_only=SHOW_NONDOMINATED_ONLY)
display_figure(figure)
plt.close(figure)

### Compare coverage at selected frontier points

For each budget, select its highest-mean nondominated point. At the largest budget, also show the highest-P05 point if it differs. Break objective ties by the other objective, lower cost and stable identity, and deduplicate identical selections. This is an explicit display rule, not a unique preference-optimal solution. The maps average sensing over that portfolio's retained draws; their utility is not calculated from the mean map. All maps use a common color scale. The plot has no connecting lines. Set `SHOW_NONDOMINATED_ONLY=False` to reveal dominated count choices as muted context points.

In [ ]:
selected_points = selected_frontier_points(frontiers)
comparison = pd.DataFrame([
    {'Budget': point['budget'], **{fleet: count for fleet, count in point['count_by_fleet'].items()},
     'Mean utility': point['utility_mean'], 'P05 utility': point['utility_p05'],
     'Std utility': point['utility_sample_std']}
    for point in selected_points
])
display(comparison.set_index('Budget'))
environment = next(iter(fleet_views.values()))[3]
selected_maps = [(point, portfolio_map(workspace, analysis, point, environment))
                 for point in selected_points]
figure = plot_frontier_maps(selected_maps, environment)
display_figure(figure)
plt.close(figure)

## 5. Finish

No CSV, image or project export is created. The following cell removes the temporary backend files. To run again after cleanup, restart from the imports/setup cell. For persistent projects, editable copies, exports and interactive satellite maps, use the web app.

In [ ]:
temporary.cleanup()